**Target:** Recovery ratio (continuous) = total_pymnt / loan_amnt. We keep only closed loans (Fully Paid, Charged Off, Default) so outcomes are observable. Alternatively, loan status with a binary default indicator for classification tasks.

**Data Split:** Loans are sorted by issue date (`issue_d`) and split chronologically into **70% train / 15% validation / 15% test**. This time-aware split prevents data leakage and look-ahead bias—the model never sees future loans when predicting past ones.

**Preprocessing** (detailed, implemented in `30_data_preprocessing.ipynb`):

1. **Missing Values**
   - **Column-level:** Drop columns with missing ratio &gt; 40% (configurable `MISSING_THRESHOLD`).
   - **Numeric features:** `SimpleImputer(strategy="median")`—median is robust to skew and outliers.
   - **Categorical features:** `SimpleImputer(strategy="most_frequent")` for modes.

2. **Outliers**
   - No explicit outlier removal or capping in the pipeline. `StandardScaler` (z-score normalization) mitigates the impact of extreme values; tree-based models are relatively robust to outliers. For highly skewed variables (e.g., `annual_inc`), percentile-based capping can be considered if needed.

3. **Feature Engineering & Column Types**
   - `issue_d` parsed to datetime; `issue_year` and `issue_month` extracted for temporal signals.
   - `term` cleaned to numeric (36 or 60 months); `emp_length` mapped to years (e.g., "10+ years" → 10); `grade` and `sub_grade` ordinal-encoded (A=1..G=7).
   - Numeric vs. categorical columns are identified automatically for separate pipelines.

4. **ColumnTransformer & Pipelines**
   - **Numeric pipeline:** `SimpleImputer(median)` → `StandardScaler`.
   - **Categorical pipeline:** `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`.
   - `ColumnTransformer` applies the appropriate pipeline to each column type; `remainder="drop"` for any unselected columns.

5. **One-Hot Encoding**
   - Categorical variables (if any remain after cleaning) are one-hot encoded so each category becomes a binary column. `handle_unknown="ignore"` ensures validation/test data with unseen categories do not cause errors.

**Modeling:** Linear regression as baseline for recovery ratio. Tree-based models (random forest, gradient boosting) for comparisons. Class weighting or stratified sampling if addressing class imbalance in classification tasks.

**Evaluation:** MSE, MAE, R² for regression; AUC-ROC, precision, recall, F1 for classification. Hyperparameters tuned via cross-validation; feature importance analyzed.